# Feature Engineering - Model II

This notebook creates a more realistic feature set for solar radiation prediction. It avoids using solar radiation component variables such as `ALLSKY_SFC_SW_DNI` and `ALLSKY_SFC_SW_DIFF` as predictors, because those variables are too close to the target `ALLSKY_SFC_SW_DWN` and may cause data leakage.

The goal is to predict solar radiation using weather variables and datetime-derived features.

In [1]:
import pandas as pd
import numpy as np

## Load Cleaned Data

In [2]:
douala = pd.read_csv("Data/Processed/Douala_Cleaned.csv")
maroua = pd.read_csv("Data/Processed/Maroua_Cleaned.csv")

print("Douala shape:", douala.shape)
print("Maroua shape:", maroua.shape)

Douala shape: (52584, 12)
Maroua shape: (52584, 12)


## Create Realistic Features

The new features are created from datetime and weather variables only. The solar variables `ALLSKY_SFC_SW_DNI` and `ALLSKY_SFC_SW_DIFF` are kept in the dataset for analysis, but they are not included in the final Model II feature list.

In [3]:
def add_model_ii_features(df, city):
    df = df.copy()

    df["datetime"] = pd.to_datetime(df["datetime"])

    # datetime-derived features
    df["hour"] = df["datetime"].dt.hour
    df["month"] = df["datetime"].dt.month
    df["day_of_year"] = df["datetime"].dt.dayofyear

    # cyclic time features help the model understand repeating daily and yearly patterns
    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
    df["day_sin"] = np.sin(2 * np.pi * df["day_of_year"] / 365)
    df["day_cos"] = np.cos(2 * np.pi * df["day_of_year"] / 365)

    # hour-based daytime indicator, so it does not use the target variable
    df["is_daytime"] = df["hour"].between(6, 18).astype(int)

    # rain indicator and rainfall transform
    df["is_rainy"] = (df["PRECTOTCORR"] > 0).astype(int)
    df["rain_log1p"] = np.log1p(df["PRECTOTCORR"])

    # wind transform to reduce the effect of large wind-speed values
    df["ws10m_log1p"] = np.log1p(df["WS10M"])

    # city-specific season definition based on Cameroon climate patterns
    if city.lower() == "douala":
        # Douala is coastal; most months are rainy, with a short dry season.
        df["season"] = np.where(df["month"].isin([12, 1, 2]), "dry", "rainy")
    elif city.lower() == "maroua":
        # Maroua is in the Far North; rainy season is shorter.
        df["season"] = np.where(df["month"].isin([6, 7, 8, 9]), "rainy", "dry")
    else:
        df["season"] = "unknown"

    # encode season for machine learning
    df["season_dry"] = (df["season"] == "dry").astype(int)
    df["season_rainy"] = (df["season"] == "rainy").astype(int)

    return df

In [4]:
douala_model_ii = add_model_ii_features(douala, "douala")
maroua_model_ii = add_model_ii_features(maroua, "maroua")

print("Douala Model II shape:", douala_model_ii.shape)
print("Maroua Model II shape:", maroua_model_ii.shape)

Douala Model II shape: (52584, 28)
Maroua Model II shape: (52584, 28)


## Model II Feature List

This feature list excludes `ALLSKY_SFC_SW_DNI` and `ALLSKY_SFC_SW_DIFF` to keep the model realistic.

In [5]:
model_ii_features = [
    "YEAR",
    "MO",
    "DY",
    "HR",
    "T2M",
    "RH2M",
    "PRECTOTCORR",
    "WS10M",
    "hour_sin",
    "hour_cos",
    "month_sin",
    "month_cos",
    "day_sin",
    "day_cos",
    "day_of_year",
    "is_daytime",
    "is_rainy",
    "rain_log1p",
    "ws10m_log1p",
    "season_dry",
    "season_rainy"
]

target = "ALLSKY_SFC_SW_DWN"

print("Model II features:")
for feature in model_ii_features:
    print("-", feature)

print("\nTarget:", target)

Model II features:
- YEAR
- MO
- DY
- HR
- T2M
- RH2M
- PRECTOTCORR
- WS10M
- hour_sin
- hour_cos
- month_sin
- month_cos
- day_sin
- day_cos
- day_of_year
- is_daytime
- is_rainy
- rain_log1p
- ws10m_log1p
- season_dry
- season_rainy

Target: ALLSKY_SFC_SW_DWN


## Inspect New Features

In [6]:
douala_model_ii[[
    "datetime",
    "T2M",
    "RH2M",
    "PRECTOTCORR",
    "WS10M",
    "hour_sin",
    "hour_cos",
    "month_sin",
    "month_cos",
    "day_of_year",
    "season",
    "is_daytime",
    "is_rainy",
    "rain_log1p",
    "ws10m_log1p"
]].head(10)

,datetime,T2M,RH2M,PRECTOTCORR,WS10M,hour_sin,hour_cos,month_sin,month_cos,day_of_year,season,is_daytime,is_rainy,rain_log1p,ws10m_log1p
0,2020-01-01 00:00:00,25.22,93.75,0.0,0.59,0.000000,1.000000e+00,0.5,0.866025,1,dry,0,0,0.0,0.463734
1,2020-01-01 01:00:00,25.25,92.37,0.0,0.37,0.258819,9.659258e-01,0.5,0.866025,1,dry,0,0,0.0,0.314811
2,2020-01-01 02:00:00,25.18,91.55,0.0,0.35,0.500000,8.660254e-01,0.5,0.866025,1,dry,0,0,0.0,0.300105
3,2020-01-01 03:00:00,25.05,91.26,0.0,0.29,0.707107,7.071068e-01,0.5,0.866025,1,dry,0,0,0.0,0.254642
4,2020-01-01 04:00:00,24.86,91.38,0.0,0.14,0.866025,5.000000e-01,0.5,0.866025,1,dry,0,0,0.0,0.131028
5,2020-01-01 05:00:00,24.74,91.30,0.0,0.11,0.965926,2.588190e-01,0.5,0.866025,1,dry,0,0,0.0,0.104360
6,2020-01-01 06:00:00,24.66,91.58,0.0,0.32,1.000000,6.123234e-17,0.5,0.866025,1,dry,1,0,0.0,0.277632
7,2020-01-01 07:00:00,25.86,96.64,0.0,0.34,0.965926,-2.588190e-01,0.5,0.866025,1,dry,1,0,0.0,0.292670
8,2020-01-01 08:00:00,27.38,77.66,0.0,0.45,0.866025,-5.000000e-01,0.5,0.866025,1,dry,1,0,0.0,0.371564
9,2020-01-01 09:00:00,29.08,69.37,0.0,0.83,0.707107,-7.071068e-01,0.5,0.866025,1,dry,1,0,0.0,0.604316


In [7]:
maroua_model_ii[[
    "datetime",
    "T2M",
    "RH2M",
    "PRECTOTCORR",
    "WS10M",
    "hour_sin",
    "hour_cos",
    "month_sin",
    "month_cos",
    "day_of_year",
    "season",
    "is_daytime",
    "is_rainy",
    "rain_log1p",
    "ws10m_log1p"
]].head(10)

,datetime,T2M,RH2M,PRECTOTCORR,WS10M,hour_sin,hour_cos,month_sin,month_cos,day_of_year,season,is_daytime,is_rainy,rain_log1p,ws10m_log1p
0,2020-01-01 00:00:00,19.53,22.81,0.0,5.19,0.000000,1.000000e+00,0.5,0.866025,1,dry,0,0,0.0,1.822935
1,2020-01-01 01:00:00,18.66,24.00,0.0,5.14,0.258819,9.659258e-01,0.5,0.866025,1,dry,0,0,0.0,1.814825
2,2020-01-01 02:00:00,17.88,25.20,0.0,5.12,0.500000,8.660254e-01,0.5,0.866025,1,dry,0,0,0.0,1.811562
3,2020-01-01 03:00:00,17.17,26.52,0.0,5.18,0.707107,7.071068e-01,0.5,0.866025,1,dry,0,0,0.0,1.821318
4,2020-01-01 04:00:00,16.50,27.93,0.0,5.24,0.866025,5.000000e-01,0.5,0.866025,1,dry,0,0,0.0,1.830980
5,2020-01-01 05:00:00,15.89,29.40,0.0,5.28,0.965926,2.588190e-01,0.5,0.866025,1,dry,0,0,0.0,1.837370
6,2020-01-01 06:00:00,15.61,30.56,0.0,5.59,1.000000,6.123234e-17,0.5,0.866025,1,dry,1,0,0.0,1.885553
7,2020-01-01 07:00:00,17.45,27.60,0.0,6.11,0.965926,-2.588190e-01,0.5,0.866025,1,dry,1,0,0.0,1.961502
8,2020-01-01 08:00:00,21.26,21.00,0.0,7.09,0.866025,-5.000000e-01,0.5,0.866025,1,dry,1,0,0.0,2.090629
9,2020-01-01 09:00:00,24.95,15.82,0.0,7.75,0.707107,-7.071068e-01,0.5,0.866025,1,dry,1,0,0.0,2.169054


## Prepare X and y

These variables can be used later in the Model II training notebook.

In [8]:
X_douala_model_ii = douala_model_ii[model_ii_features]
y_douala_model_ii = douala_model_ii[target]

X_maroua_model_ii = maroua_model_ii[model_ii_features]
y_maroua_model_ii = maroua_model_ii[target]

print("Douala X shape:", X_douala_model_ii.shape)
print("Douala y shape:", y_douala_model_ii.shape)
print("Maroua X shape:", X_maroua_model_ii.shape)
print("Maroua y shape:", y_maroua_model_ii.shape)

Douala X shape: (52584, 21)
Douala y shape: (52584,)
Maroua X shape: (52584, 21)
Maroua y shape: (52584,)


## Save Model II Feature Datasets

In [ ]:
douala_model_ii.to_csv("Data/Processed/Douala_Features_Model_II.csv", index=False)
maroua_model_ii.to_csv("Data/Processed/Maroua_Features_Model_II.csv", index=False)

print("Saved: Data/Processed/Douala_Features_Model_II.csv")
print("Saved: Data/Processed/Maroua_Features_Model_II.csv")

Saved: Data/Processed/Douala_Features_Model_II.csv
Saved: Data/Processed/Maroua_Features_Model_II.csv


: 

## Model II Notes

- `ALLSKY_SFC_SW_DNI` and `ALLSKY_SFC_SW_DIFF` are excluded from `model_ii_features` to avoid leakage.
- `hour_sin` and `hour_cos` help the model understand the daily solar cycle.
- `month_sin`, `month_cos`, `day_sin`, and `day_cos` help the model understand seasonal cycles.
- `is_daytime` is based on hour, not the target variable, so it is safe for forecasting.
- `is_rainy`, `rain_log1p`, and `ws10m_log1p` help represent rainfall and wind behavior more clearly.
- `season_dry` and `season_rainy` are based on city-specific climate patterns.